# Fooocus Colab Edition 2.6

El preset `default` no descarga modelos. Indica una URL de Hugging Face y la celda la descarga aparte con aria2c mostrando progreso continuo.

La instalación aprovecha Python/Torch/CUDA preinstalados en Colab.

In [ ]:
# @title Descargar modelo y arrancar Fooocus 2.6
BRANCH = 'main'  # @param {type:"string"}
MODEL_URL = 'https://huggingface.co/lllyasviel/fav_models/resolve/main/fav/juggernautXL_v8Rundiffusion.safetensors'  # @param {type:"string"}
MODEL_FILENAME = 'model.safetensors'  # @param {type:"string"}
TUNEL = 'cloudflare'  # @param ["cloudflare", "gradio"]
CACHEAR_MODELOS_EN_DRIVE = False  # @param {type:"boolean"}
ARGUMENTOS_EXTRA = ''  # @param {type:"string"}

import os, re, shlex, shutil, subprocess, sys, threading, queue, time
REPO = 'https://github.com/deleonramiro085/Fooocus.git'
WORKDIR = '/content/Fooocus'
PORT = 7865
DRIVE_CACHE = '/content/drive/MyDrive/Fooocus/models'
SUBDIRS = ['checkpoints','loras','inpaint','controlnet','clip_vision','upscale_models','vae','vae_approx','sam','safety_checker']
CF_DEB = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'NO DETECTADA', flush=True)
if not gpu: raise RuntimeError('Activa una GPU en Entorno de ejecucion > Cambiar tipo de entorno.')

if not os.path.isdir(os.path.join(WORKDIR, '.git')):
    shutil.rmtree(WORKDIR, ignore_errors=True)
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,WORKDIR], check=True)
else:
    print('Actualizando repositorio...', flush=True)
    subprocess.run(['git','-C',WORKDIR,'fetch','origin',BRANCH,'--depth','1'], check=False)
    status = subprocess.run(['git','-C',WORKDIR,'status','--porcelain'], capture_output=True, text=True).stdout.strip()
    if not status:
        subprocess.run(['git','-C',WORKDIR,'reset','--hard',f'origin/{BRANCH}'], check=True)
os.chdir(WORKDIR)

print('Instalando aria2...', flush=True)
if shutil.which('aria2c') is None:
    subprocess.run(['apt-get','update','-qq'], check=False)
    subprocess.run(['apt-get','install','-y','-qq','aria2'], check=True)
print('aria2c listo:', shutil.which('aria2c'), flush=True)

if CACHEAR_MODELOS_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    for sub in SUBDIRS:
        target = os.path.join(DRIVE_CACHE, sub); os.makedirs(target, exist_ok=True)
        local = os.path.join(WORKDIR,'models',sub)
        if not os.path.islink(local):
            shutil.rmtree(local, ignore_errors=True); os.symlink(target, local)

if not MODEL_URL.strip():
    raise ValueError('Pega MODEL_URL de Hugging Face antes de ejecutar.')
model_dir = os.path.join(WORKDIR, 'models', 'checkpoints'); os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, MODEL_FILENAME)
if os.path.isfile(model_path) and os.path.getsize(model_path) > 1000000000:
    print('Modelo ya presente:', model_path, flush=True)
else:
    print('Descargando modelo con aria2c: 16 conexiones', flush=True)
    cmd_dl = ['aria2c','--console-log-level=notice','--summary-interval=5','--continue=true','--allow-overwrite=true','--auto-file-renaming=false','--check-integrity=true','--max-connection-per-server=16','--split=16','--min-split-size=1M','--max-tries=5','--retry-wait=3','--timeout=60','--dir',model_dir,'--out',MODEL_FILENAME,MODEL_URL]
    subprocess.run(cmd_dl, check=True)
    if not os.path.isfile(model_path) or os.path.getsize(model_path) <= 1000000000: raise RuntimeError('El checkpoint no parece completo; revisa la URL.')
print('Modelo listo:', model_path, flush=True)

tunel_url = None
if TUNEL == 'cloudflare':
    if shutil.which('cloudflared') is None:
        subprocess.run(['wget','-q','-O','/content/cloudflared.deb',CF_DEB], check=False)
        subprocess.run(['dpkg','-i','/content/cloudflared.deb'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if shutil.which('cloudflared'):
        p = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url',f'http://127.0.0.1:{PORT}'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        q = queue.Queue()
        threading.Thread(target=lambda: [q.put(x) for x in p.stdout], daemon=True).start()
        deadline = time.monotonic() + 45
        while time.monotonic() < deadline and tunel_url is None:
            try:
                m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', q.get(timeout=1))
                if m: tunel_url = m.group(0)
            except queue.Empty: pass
        if tunel_url: print('URL PUBLICA:', tunel_url, flush=True)
        else: print('Cloudflare no respondio; se usa --share.', flush=True); p.kill()

cmd = [sys.executable,'-u','entry_with_update.py','--skip-update','--preset','default','--disable-preset-download','--disable-analytics','--port',str(PORT)]
cmd += ['--listen','127.0.0.1'] if tunel_url else ['--share']
cmd += shlex.split(ARGUMENTOS_EXTRA)
print('Ejecutando:', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=False)


In [ ]:
# @title Diagnostico
import importlib.metadata as md, platform, shutil
print('python', platform.python_version())
try:
 import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda); print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU')
except Exception as e: print('torch:', e)
print('aria2c', shutil.which('aria2c') or 'no instalado')
print('cloudflared', shutil.which('cloudflared') or 'no instalado')
for p in ('gradio','gradio_client','numpy','transformers','huggingface_hub','pydantic','fastapi','starlette','websockets','pygit2'):
 try: print(p, md.version(p))
 except Exception: print(p, 'no instalado')


## Notas

Pega la URL directa del archivo en `MODEL_URL`. Si el archivo se llama distinto, pon ese nombre en `MODEL_FILENAME`; el nombre debe coincidir con el checkpoint que Fooocus cargará. La primera ejecución instala únicamente lo necesario y cada etapa imprime progreso.